# 🧠 FastMemory + LangChain: The Logical-Grounding Loop

Traditional RAG sends random chunks of text to an LLM, leading to "hallucinated synthesis" where the AI tries to guess the answer. 

**FastMemory** provides a **Logical Block** (Action, Logic, Data, Access, Event). In this notebook, we use **LangChain** to ground the LLM's reasoning within these deterministic boundaries.

## 🏗️ Phase 1: The FastMemory Builder

We start by clustering our unstructured documentation into a Topological graph.

In [ ]:
import fastmemory
import json
import os

# Sample Action-Topology Format (ATF) documents
atf_docs = """
## [Category: Banking_Security]
### [ID: wire_transfer_logic]
**Action:** Authorize_International_Wire
**Logic:** Calculate Risk_Score (0-100). If amount > $10,000 and the destination is 'High_Risk', trigger MFA_SMS.
**Data_Connections:** db:swift_vault, svc:mfa_provider
**Access:** Role_Treasury_Ops
**Events:** Fraud_Trigger_Logged
"""

# Construct the Federated Memory
topology_json = fastmemory.process_markdown(atf_docs)
with open("graph_db.json", "w") as f:
    f.write(topology_json)

print("✅ Memory Graph constructed and persisted.")

## 🕵️ Phase 2: LangChain Discovery (Grounding Architecture)

In this section, we create a function to retrieve the **exact logical block** needed for a query. This replaces the standard semantic similarity search with **Topological Grounding**.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def get_topological_context(query_keyword):
    """
    Retrieves the isolated logical block from the Topology Graph.
    This simulates the 'Agentic Query' against the Graph DB.
    """
    with open("graph_db.json", "r") as f:
        graph = json.load(f)
    
    keyword = query_keyword.lower()
    for cluster in graph:
        if keyword in json.dumps(cluster).lower():
            # Return the block with cleaned logic formatting for the LLM
            return json.dumps(cluster, indent=2)
    
    return "Out of Topological Bound: No matching logic available."

print("Topological Retriever Ready!")

## 🤖 Phase 3: The 'Logic-Locked' LLM Query

Here we build the LangChain loop. We instruct the LLM to follow **strict adherence** to the `Logic` and `Access` nodes found in the FastMemory block. 

If the code tries to hallucinate a rule not in the Topology, it will be physically blocked by our grounding prompt.

In [ ]:
# -------------------------------------------------------------------------
# PRODUCTION LLM SETUP (OPTIONAL)
# -------------------------------------------------------------------------
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o", temperature=0)
# -------------------------------------------------------------------------

# 1. Define the Grounding Prompt Template
template = """
SYSTEM: You are a Deterministic Engineering Architect.
You have access to a logical segment of the enterprise TOPOLOGY ARCHITECTURE.
Your response MUST adhere EXCLUSIVELY to the logic, data connections, and access rules provided below.

TOPOLOGY BLOCK:
{topology_block}

USER REQUEST: {question}

INSTRUCTIONS:
1. If the user's request exceeds the 'Logic' or 'Access' rules, inform them of the boundary restriction.
2. Cite the [ID] of the block used for this answer.
3. Do not synthesize information from baseline memory; use only this 'Horizontal Layer of Truth'.
"""
prompt = ChatPromptTemplate.from_template(template)

# 2. Build the Grounded Chain
from langchain_community.llms.fake import FakeListLLM
llm = FakeListLLM(responses=["Based on [ID: wire_transfer_logic], you must trigger MFA_SMS for any amount over $10,000 for high-risk destinations. This requires Role_Treasury_Ops access."])

chain = (
    {"topology_block": lambda x: get_topological_context(x["keyword"]), "question": RunnablePassthrough(), "keyword": lambda x: x["keyword"]}
    | prompt
    | llm
    | StrOutputParser()
)

# 3. Execute the Grounded Query
result = chain.invoke({"question": "What are the rules for $15,000 international wires?", "keyword": "wire"})

print("--- LLM GROUNDED RESPONSE ---")
print(result)

## 🚀 Why this is the Future of AI Engineering

- **Isolation**: The LLM never sees the rest of the file. It is locked into a single logical cluster.
- **Deterministic Citation**: Every answer is traceable to a specific ontological [ID].
- **Access Aware**: The LLM knows the `Role_Treasury_Ops` requirement because it was in the retrieved block. 

**FastMemory + LangChain = Zero Hallucinations. High Velocity.** 🛡️💻🧠

[Join the Enterprise Waitlist](https://fastbuilder.ai)